# Gold Layer — checks (design 1.3 / 1.4 / 1.5)

Run after every Gold load. Each check is a query that must return **0**. Any failure raises, so the daily job stops before `agg_restaurant_ranking` is rebuilt.

- **Grain:** one row per natural key in every dimension, one row per grain in every fact; exactly one Unknown (−1) row per dimension.
- **Orphans:** every fact key exists in its dimension; no fact key is NULL.
- **No counters in dims:** counters belong to the snapshot facts.
- **Read grain (since 2026-11-02):** `fact_video_snapshot` / `fact_channel_snapshot` = one row per id × `snapshot_ts` (comments: `dim_comment`, one row per `comment_id`, since 2026-11-02); the per-day serving tables `agg_video_daily` / `agg_channel_daily` (serving schema since 2026-11-02) = one row per id × day.
- **Dense snapshots (warning):** from `DENSE_FROM` (first day of the daily refresh) every day should have a snapshot. A missed day cannot be backfilled (the YouTube API has no history), and `views_gained` after a gap still adds up correctly, so a gap is reported but does not stop the job. While the schedule is paused and the job runs by hand, gaps are expected.
- **Google reviews** (once built): one row per `review_id`, rating 1–5, keys exist in `dim_restaurant` / `dim_date`.
- **Restaurant mentions** (once built): `SUM(mention_weight) = 1` per video, one current version per `restaurant_id`.

Warnings (printed, not failing): `view_count` dropping between two snapshots; days without a snapshot since `DENSE_FROM`.


In [0]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("serving", "prod.serving")
SERVING = dbutils.widgets.get("serving")  # serving layer: agg_* tables read by the dashboard / app / chat (2026-11-02)

DENSE_FROM = "2026-10-28"  # dense daily snapshots start here; 2026-10-27 was never captured (job skipped), earlier history is sparse


In [0]:
def table_exists(name):
    return spark.catalog.tableExists(f"{GOLD}.{name}")


def tbl(name):
    """agg_* tables live in the serving schema (since 2026-11-02), the rest in Gold."""
    return f"{SERVING}.{name}" if name.startswith("agg_") else f"{GOLD}.{name}"


checks = {
    # grain: dimensions
    "dim_date_scd0: duplicate date_key": f"SELECT COUNT(*) - COUNT(DISTINCT date_key) FROM {GOLD}.dim_date_scd0",
    "dim_video_scd1: duplicate video_id": f"SELECT COUNT(*) - COUNT(DISTINCT video_id) FROM {GOLD}.dim_video_scd1 WHERE video_key <> -1",
    "dim_channel_scd1: duplicate channel_id": f"SELECT COUNT(*) - COUNT(DISTINCT channel_id) FROM {GOLD}.dim_channel_scd1 WHERE channel_key <> -1",
    "dim_commenter_scd1: duplicate commenter_key": f"SELECT COUNT(*) - COUNT(DISTINCT commenter_key) FROM {GOLD}.dim_commenter_scd1",
    "dim_keyword_scd1: duplicate keyword_normalized": f"SELECT COUNT(*) - COUNT(DISTINCT keyword_normalized) FROM {GOLD}.dim_keyword_scd1 WHERE keyword_key <> -1",
    # grain: facts
    "fact_video_snapshot_poc: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT snapshot_ts, video_key) FROM {GOLD}.fact_video_snapshot_poc WHERE video_key <> -1",
    "fact_channel_snapshot_poc: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT snapshot_ts, channel_key) FROM {GOLD}.fact_channel_snapshot_poc WHERE channel_key <> -1",
    "agg_video_daily: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT snapshot_date_key, video_key) FROM {SERVING}.agg_video_daily",
    "agg_channel_daily: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT snapshot_date_key, channel_key) FROM {SERVING}.agg_channel_daily",
    "dim_comment_scd1: duplicate comment_id": f"SELECT COUNT(*) - COUNT(DISTINCT comment_id) FROM {GOLD}.dim_comment_scd1 WHERE comment_key <> -1",
    "fact_search_hit_poc: duplicate grain": f"SELECT COUNT(*) - COUNT(DISTINCT fetched_at, keyword_key, video_key) FROM {GOLD}.fact_search_hit_poc",
    # Unknown members
    **{
        f"{dim}: not exactly one Unknown row": f"SELECT ABS(COUNT_IF({key} = -1) - 1) FROM {GOLD}.{dim}"
        for dim, key in [("dim_date_scd0", "date_key"), ("dim_video_scd1", "video_key"), ("dim_channel_scd1", "channel_key"),
                         ("dim_commenter_scd1", "commenter_key"), ("dim_comment_scd1", "comment_key"),
                         ("dim_keyword_scd1", "keyword_key")]
    },
}

# dense snapshots: warning only (a missed day cannot be backfilled; views_gained after a gap still adds up)
missing_days_sql = {
    fact: f"""
        SELECT collect_list(CAST(d.full_date AS STRING)) FROM (
          SELECT full_date FROM {GOLD}.dim_date_scd0 d
          WHERE d.full_date BETWEEN DATE'{DENSE_FROM}' AND (SELECT to_date(CAST(MAX(snapshot_date_key) AS STRING), 'yyyyMMdd') FROM {tbl(fact)})
            AND d.date_key NOT IN (SELECT DISTINCT snapshot_date_key FROM {tbl(fact)})
          ORDER BY full_date) d"""
    for fact in ("agg_video_daily", "agg_channel_daily")
}

# orphans and NULL keys: (fact, fact column, dimension, dimension key)
FACT_KEYS = [
    ("fact_video_snapshot_poc", "snapshot_date_key", "dim_date_scd0", "date_key"),
    ("fact_video_snapshot_poc", "video_key", "dim_video_scd1", "video_key"),
    ("fact_video_snapshot_poc", "channel_key", "dim_channel_scd1", "channel_key"),
    ("fact_channel_snapshot_poc", "snapshot_date_key", "dim_date_scd0", "date_key"),
    ("fact_channel_snapshot_poc", "channel_key", "dim_channel_scd1", "channel_key"),
    ("agg_video_daily", "snapshot_date_key", "dim_date_scd0", "date_key"),
    ("agg_video_daily", "video_key", "dim_video_scd1", "video_key"),
    ("agg_video_daily", "channel_key", "dim_channel_scd1", "channel_key"),
    ("agg_channel_daily", "snapshot_date_key", "dim_date_scd0", "date_key"),
    ("agg_channel_daily", "channel_key", "dim_channel_scd1", "channel_key"),
    # dim_comment_scd1 outrigger keys to dim_video_scd1 / dim_commenter_scd1
    ("dim_comment_scd1", "video_key", "dim_video_scd1", "video_key"),
    ("dim_comment_scd1", "commenter_key", "dim_commenter_scd1", "commenter_key"),
    ("fact_search_hit_poc", "fetched_date_key", "dim_date_scd0", "date_key"),
    ("fact_search_hit_poc", "keyword_key", "dim_keyword_scd1", "keyword_key"),
    ("fact_search_hit_poc", "video_key", "dim_video_scd1", "video_key"),
    ("fact_search_hit_poc", "channel_key", "dim_channel_scd1", "channel_key"),
]

# restaurant tables are built later (design step 6); check them once they exist
if table_exists("dim_restaurant_scd2"):
    # closed restaurants (left the source) have no current version, so the rule is "at most one"
    checks["dim_restaurant_scd2: more than one current version per restaurant_id"] = f"""
        SELECT COUNT(*) FROM (SELECT restaurant_id FROM {GOLD}.dim_restaurant_scd2
                              GROUP BY restaurant_id HAVING COUNT_IF(current_flag = 'Y') > 1)"""
    checks["dim_restaurant_scd2: mentioned restaurant without a current version"] = f"""
        SELECT COUNT(*) FROM {GOLD}.fact_video_restaurant_mention_poc f
        LEFT ANTI JOIN {GOLD}.dim_restaurant_scd2 c ON c.restaurant_id = f.restaurant_id AND c.current_flag = 'Y'"""
    # v2 (2026-11-02): key = restaurant_id + start_date, no surrogate key
    checks["dim_restaurant_scd2: duplicate (restaurant_id, start_date)"] = f"SELECT COUNT(*) - COUNT(DISTINCT restaurant_id, start_date) FROM {GOLD}.dim_restaurant_scd2"
    checks["dim_restaurant_scd2: NULL restaurant_id"] = f"SELECT COUNT_IF(restaurant_id IS NULL) FROM {GOLD}.dim_restaurant_scd2"
    checks["dim_restaurant_scd2: not exactly one Unknown row"] = f"SELECT ABS(COUNT_IF(restaurant_id = 'unknown') - 1) FROM {GOLD}.dim_restaurant_scd2"
    checks["dim_restaurant_scd2: start_date > end_date"] = f"SELECT COUNT_IF(start_date > end_date) FROM {GOLD}.dim_restaurant_scd2"
    checks["dim_restaurant_scd2: ward_code not in the GSO list"] = f"""
        SELECT COUNT_IF(ward_code IS NOT NULL AND NOT ward_code RLIKE '^[0-9]{{5}}$') FROM {GOLD}.dim_restaurant_scd2"""
    checks["dim_restaurant_scd2: district_code without a name"] = f"""
        SELECT COUNT_IF(district_code IS NOT NULL AND district_name IS NULL) FROM {GOLD}.dim_restaurant_scd2"""
    checks["dim_restaurant_scd2: open_time / close_time not HH:mm"] = f"""
        SELECT COUNT_IF(NOT (open_time RLIKE '^[0-2][0-9]:[0-5][0-9]$') OR NOT (close_time RLIKE '^[0-2][0-9]:[0-5][0-9]$'))
        FROM {GOLD}.dim_restaurant_scd2"""
if table_exists("dim_google_review_scd1"):   # Google reviews from Place Details (dimension since 2026-11-02)
    checks["dim_google_review_scd1: duplicate review_id"] = f"SELECT COUNT(*) - COUNT(DISTINCT review_id) FROM {GOLD}.dim_google_review_scd1 WHERE review_key <> -1"
    checks["dim_google_review_scd1: not exactly one Unknown row"] = f"SELECT ABS(COUNT_IF(review_key = -1) - 1) FROM {GOLD}.dim_google_review_scd1"
    checks["dim_google_review_scd1: rating outside 1-5"] = f"SELECT COUNT_IF(rating NOT BETWEEN 1 AND 5) FROM {GOLD}.dim_google_review_scd1 WHERE review_key <> -1"
    FACT_KEYS += [("dim_google_review_scd1", "restaurant_id", "dim_restaurant_scd2", "restaurant_id"),
                  ("dim_google_review_scd1", "publish_date_key", "dim_date_scd0", "date_key")]
if table_exists("fact_video_restaurant_mention_poc"):
    checks["fact_video_restaurant_mention_poc: duplicate grain"] = f"SELECT COUNT(*) - COUNT(DISTINCT video_key, restaurant_id) FROM {GOLD}.fact_video_restaurant_mention_poc"
    checks["fact_video_restaurant_mention_poc: SUM(mention_weight) <> 1 per video"] = f"""
        SELECT COUNT(*) FROM (SELECT video_key FROM {GOLD}.fact_video_restaurant_mention_poc
                              GROUP BY video_key HAVING ABS(SUM(mention_weight) - 1) > 1e-9)"""
    FACT_KEYS += [("fact_video_restaurant_mention_poc", "video_key", "dim_video_scd1", "video_key"),
                  ("fact_video_restaurant_mention_poc", "restaurant_id", "dim_restaurant_scd2", "restaurant_id")]
if table_exists("dim_dish_scd1"):   # dishes named in the reviews (2026-11-02), child of dim_restaurant_scd2
    checks["dim_dish_scd1: duplicate (restaurant_id, dish_name_normalized)"] = f"SELECT COUNT(*) - COUNT(DISTINCT restaurant_id, dish_name_normalized) FROM {GOLD}.dim_dish_scd1"
    checks["dim_dish_scd1: price outside 5,000-5,000,000 VND"] = f"SELECT COUNT_IF(price_vnd NOT BETWEEN 5000 AND 5000000) FROM {GOLD}.dim_dish_scd1"
    FACT_KEYS += [("dim_dish_scd1", "restaurant_id", "dim_restaurant_scd2", "restaurant_id")]

for fact, col, dim, key in FACT_KEYS:
    checks[f"{fact}.{col}: NULL"] = f"SELECT COUNT(*) FROM {tbl(fact)} WHERE {col} IS NULL"
    checks[f"{fact}.{col}: orphan (not in {dim})"] = f"""
        SELECT COUNT(*) FROM {tbl(fact)} f LEFT ANTI JOIN {GOLD}.{dim} d ON d.{key} = f.{col}"""

# no counters in dimensions
for dim in ["dim_video_scd1", "dim_channel_scd1"]:
    counters = [c for c in spark.table(f"{GOLD}.{dim}").columns if c.endswith("_count")]
    checks[f"{dim}: holds counters {counters}"] = f"SELECT {len(counters)}"


In [0]:
# --- Run ---
failures = []
for name, sql in checks.items():
    bad = spark.sql(sql).collect()[0][0] or 0
    print(("❌" if bad else "✅") + f" {name}: {bad}")
    if bad:
        failures.append(f"{name}: {bad}")

drops = spark.sql(f"SELECT COUNT(*) FROM {SERVING}.agg_video_daily WHERE views_gained < 0").collect()[0][0]
print(("⚠️" if drops else "✅") + f" agg_video_daily: rows where view_count dropped: {drops} (warning only)")
for fact, sql in missing_days_sql.items():
    missing = spark.sql(sql).collect()[0][0] or []
    print(("⚠️" if missing else "✅") + f" {fact}: days without a snapshot since {DENSE_FROM}: {missing} (warning only)")

if failures:
    raise AssertionError(f"{len(failures)} Gold check(s) failed:\n" + "\n".join(failures))
print(f"\n--- All {len(checks)} Gold checks passed ---")
